In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Load data (change the paths if you're on Kaggle: /kaggle/input/<competition-name>/)
train = pd.read_csv("/kaggle/input/competitions/mle-practice-test/train.csv")
test = pd.read_csv("/kaggle/input/competitions/mle-practice-test/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/mle-practice-test/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Missing values in train:", train.isnull().sum().sum())
print("Class balance:\n", train["label"].value_counts(normalize=True))

# 2. Separate features and target
X = train.drop(columns=["id", "label"])
y = train["label"]
X_test = test.drop(columns=["id"])

# 3. Find numeric and categorical columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("Numeric columns:", len(num_cols), "| Categorical columns:", len(cat_cols))

# 4. Preprocessing
# numeric: fill missing with median, then scale
# categorical: fill missing with most frequent value, then one-hot encode
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

# 5. Train/validation split to check how the model does
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Try a few models and compare
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=12,
                                            random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200,
                                                    learning_rate=0.05,
                                                    max_depth=3,
                                                    random_state=42),
}

results = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    acc = accuracy_score(y_val, pipe.predict(X_val))
    results[name] = acc
    print(f"{name}: validation accuracy = {acc:.4f}")

# 7. Pick the best model
best_name = max(results, key=results.get)
print("\nBest model:", best_name)

# 8. Quick cross-validation on the best model to make sure it's stable
best_pipe = Pipeline([("prep", preprocess), ("model", models[best_name])])
cv_scores = cross_val_score(best_pipe, X, y, cv=5, scoring="accuracy")
print("5-fold CV accuracy: %.4f (+/- %.4f)" % (cv_scores.mean(), cv_scores.std()))

# 9. Retrain on the full training data and predict on test
best_pipe.fit(X, y)
test_preds = best_pipe.predict(X_test)

# 10. Create submission file in the required format
submission = pd.DataFrame({
    "id": test["id"],
    "label": test_preds.astype(int),
})
submission = submission[sample.columns]   # keep the exact column order of the sample
submission.to_csv("submission.csv", index=False)

print("\nSubmission saved:", submission.shape)
print(submission.head())

Train shape: (3200, 35)
Test shape: (800, 34)
Missing values in train: 2088
Class balance:
 label
0    0.650937
1    0.349062
Name: proportion, dtype: float64
Numeric columns: 23 | Categorical columns: 10
Logistic Regression: validation accuracy = 0.9422
Random Forest: validation accuracy = 0.9750
Gradient Boosting: validation accuracy = 0.9938

Best model: Gradient Boosting
5-fold CV accuracy: 0.9959 (+/- 0.0038)

Submission saved: (800, 2)
     id  label
0   555      0
1  3491      1
2   527      1
3  3925      1
4  2989      0
